In [5]:
import requests
import zipfile
import io
import pandas as pd
import os
from datetime import datetime

# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_mcap"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)
os.makedirs("/lakehouse/default/Files/data/raw/logs", exist_ok=True)

try:

    # today's date
    today = datetime.today()

    pr_date = today.strftime("%d%m%y")
    bhav_date = today.strftime("%Y%m%d")

    # NSE daily package URL
    url = (
        "https://www.nseindia.com/api/"
        "zipem?fileURLs=%5B"
        "%22https%3A%2F%2Fnsearchives.nseindia.com"
        f"%2Farchives%2Fequities%2Fbhavcopy%2Fpr%2FPR{pr_date}.zip%22,"
        "%22https%3A%2F%2Fnsearchives.nseindia.com"
        f"%2Fcontent%2Fcm%2FBhavCopy_NSE_CM_0_0_0_{bhav_date}_F_0000.csv.zip%22"
        "%5D&type=Daily"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.nseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    # open outer zip
    outer_zip = zipfile.ZipFile(
        io.BytesIO(response.content)
    )

    # find PR zip
    pr_file = None

    for file in outer_zip.namelist():

        if file.upper().startswith("PR"):
            pr_file = file
            break

    if pr_file is None:
        raise Exception("PR zip not found")

    # open PR zip
    pr_bytes = outer_zip.read(pr_file)

    pr_zip = zipfile.ZipFile(
        io.BytesIO(pr_bytes)
    )

    # find market cap file
    mcap_file = None

    for file in pr_zip.namelist():

        if "mcap" in file.lower():
            mcap_file = file
            break

    if mcap_file is None:
        raise Exception("Market cap file not found")

    # read csv
    mcap_df = pd.read_csv(
        pr_zip.open(mcap_file)
    )

    # save file
    save_path = os.path.join(
        save_dir,
        mcap_file
    )

    mcap_df.to_csv(
        save_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(mcap_df)
    message = f"Saved: {mcap_file}"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)

# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_mcap",
    "status": status,
    "rows": rows,
    "message": message
}])

# append log
if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row

log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, 76ea4ebd-f671-47b1-8cee-188f10ee5161, 7, Finished, Available, Finished, False)

SUCCESS
Rows: 2947
Saved: mcap09062026.csv
